### STEP 1: Imports, Seed, Device

In [1]:
import re, math, random
import numpy as np
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from datasets import load_dataset

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


### STEP 2: Load dataset

In [2]:
dataset = load_dataset("wikitext", "wikitext-103-v1")
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


Dataset Reference:

The WikiText-103 dataset is a large-scale natural language corpus derived from high-quality articles extracted from Wikipedia. It was introduced by Salesforce Research in the paper “WikiText: A Large-Scale Dataset for Language Modeling” (2016) with the goal of supporting research in language modeling and long-term dependency learning.

The dataset contains approximately 103 million tokens and consists of long, complete Wikipedia articles, rather than short or artificially segmented text. Unlike many web-scraped corpora, WikiText-103 preserves natural punctuation, capitalization, rare words, and document structure, making it more challenging and realistic for training and evaluating sequence models such as LSTM-based language models and transformer architectures.

WikiText-103 is publicly distributed through the Hugging Face Datasets library, which provides standardized train, validation, and test splits, enabling reproducible experimentation. Due to its scale and linguistic quality, WikiText-103 has been widely used as a benchmark dataset for neural language modeling, perplexity evaluation, and pretraining experiments.

In [3]:
# -------------------------------
# Set a fixed random seed value
# -------------------------------
SEED = 1234

# Fix Python built-in random generator
# This ensures same random results every time we run the code
random.seed(SEED)

# Fix NumPy random generator
# Useful when NumPy is used for shuffling, sampling, etc.
np.random.seed(SEED)

# Fix PyTorch random generator
# Ensures reproducibility in PyTorch operations
torch.manual_seed(SEED)


# ------------------------------------------------
# Use only a subset of the dataset (IMPORTANT)
# ------------------------------------------------
# Shuffle the training dataset using the fixed seed
# Then select only first 100,000 sentences
# This reduces training time and memory usage
sentences = (
    dataset["train"]
    .shuffle(seed=SEED)
    .select(range(100000))
)["text"]


# --------------------------------
# Text preprocessing steps
# --------------------------------

# Convert all text to lowercase and remove leading/trailing spaces
text = [s.lower().strip() for s in sentences]

# Remove punctuation marks: . , ! ? -
# Using regular expressions (regex)
text = [re.sub(r"[.,!?\\-]", "", s) for s in text]

# Replace multiple spaces with a single space
# Also remove extra spaces at start/end
text = [re.sub(r"\s+", " ", s).strip() for s in text]

# Remove empty sentences (if any)
text = [s for s in text if s != ""]


# --------------------------------
# Display dataset information
# --------------------------------

# Print total number of cleaned sentences
print("Total sentences:", len(text))

# Print first 150 characters of one example sentence
print("Example:", text[0][:150])


Total sentences: 64753
Example: = al @@ maʿarri =


### Step 3: Build vocabulary

In [4]:
# Define special tokens used in BERT-style models
special_tokens = ['[PAD]', '[CLS]', '[SEP]', '[MASK]', '[UNK]']

# Create a list of unique words from the text corpus
word_list = list(set(" ".join(text).split()))

# Initialize word-to-id dictionary with special tokens
word2id = {tok: i for i, tok in enumerate(special_tokens)}
offset = len(word2id)  # starting index for normal words

# Assign unique IDs to each word in the vocabulary
for i, w in enumerate(word_list):
    if w not in word2id:
        word2id[w] = offset + i

# Create reverse mapping: id to word
id2word = {v: k for k, v in word2id.items()}

# Total vocabulary size
vocab_size = len(word2id)

# Store IDs of special tokens for easy access
PAD_ID  = word2id['[PAD]']
CLS_ID  = word2id['[CLS]']
SEP_ID  = word2id['[SEP]']
MASK_ID = word2id['[MASK]']
UNK_ID  = word2id['[UNK]']

# Print vocabulary size
print("Vocab size:", vocab_size)


Vocab size: 110759


### Step4:  Tokenize sentences

In [5]:
# Convert each sentence into a list of token IDs
token_list = []
for s in text:
    token_list.append([word2id.get(w, UNK_ID) for w in s.split()])

# Print number of tokenized sentences
print("Tokenized sentences:", len(token_list))


Tokenized sentences: 64753


In [6]:
#take a look at token_list
token_list[:2]

[[105096, 75636, 109883, 106097, 105096],
 [46062,
  82235,
  83751,
  2925,
  29487,
  38310,
  18209,
  28151,
  68561,
  38310,
  45079,
  21993,
  31095,
  95390,
  71037,
  50541,
  14226,
  41259,
  82235,
  46062,
  83751,
  2925,
  71335,
  78269,
  88629,
  21993,
  33902,
  46062,
  17564,
  105694,
  93207,
  47326,
  53474,
  45728,
  90289,
  46062,
  17564,
  105694,
  57625,
  4424,
  21207,
  61723,
  72167,
  20063,
  98567,
  57943,
  57659,
  46062,
  17129,
  105694,
  46062,
  1116,
  20946,
  107693,
  46062,
  20370,
  87477,
  89941,
  16082,
  61723,
  46062,
  20370,
  34482,
  105694,
  82235,
  60336,
  87027,
  46062,
  83751,
  71335,
  69827,
  33687,
  21993,
  46062,
  89423,
  105694,
  102005,
  98519,
  70596,
  50541,
  32257,
  88222,
  101537,
  105694,
  84722,
  45728,
  88222,
  104494,
  105694,
  6029,
  18043,
  46062,
  35041,
  98519,
  32764,
  50541,
  46062,
  7650,
  105694,
  102005,
  13689,
  63463,
  105694,
  46062,
  67795,
  899

Hyperparameters

In [7]:
batch_size = 16
max_len = 128
max_mask = 20

n_layers = 4
n_heads = 4
d_model = 256
d_ff = d_model * 4
n_segments = 2


### Step 5: Dataloader

In [8]:
def make_batch():
    batch = []
    positive = negative = 0
    half = batch_size // 2

    while positive < half or negative < half:
        idx_a, idx_b = np.random.randint(len(token_list), size=2)

        tokens_a = token_list[idx_a][:50]
        tokens_b = token_list[idx_b][:50]

        input_ids = [CLS_ID] + tokens_a + [SEP_ID] + tokens_b + [SEP_ID]
        segment_ids = [0]*(len(tokens_a)+2) + [1]*(len(tokens_b)+1)

        # NSP label
        if idx_b == idx_a + 1 and positive < half:
            isNext = 1
            positive += 1
        elif idx_b != idx_a + 1 and negative < half:
            isNext = 0
            negative += 1
        else:
            continue

        # MLM
        cand_pos = [i for i,t in enumerate(input_ids) if t not in (CLS_ID, SEP_ID)]
        random.shuffle(cand_pos)
        n_pred = min(max_mask, max(1, int(len(input_ids)*0.15)))
        masked_pos = cand_pos[:n_pred]

        masked_tokens = []
        for pos in masked_pos:
            masked_tokens.append(input_ids[pos])
            input_ids[pos] = MASK_ID

        # padding
        pad_len = max_len - len(input_ids)
        input_ids += [PAD_ID]*pad_len
        segment_ids += [0]*pad_len

        masked_tokens += [PAD_ID]*(max_mask-len(masked_tokens))
        masked_pos += [0]*(max_mask-len(masked_pos))

        batch.append([input_ids, segment_ids, masked_tokens, masked_pos, isNext])

    return batch


In [9]:
batch = make_batch()

In [10]:
#len of batch
len(batch)

16

In [11]:
#we can deconstruct using map and zip
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))
input_ids.shape, segment_ids.shape, masked_tokens.shape, masked_pos.shape, isNext.shape

(torch.Size([16, 128]),
 torch.Size([16, 128]),
 torch.Size([16, 20]),
 torch.Size([16, 20]),
 torch.Size([16]))

### Step 6: model

#### 6.1 Embedding

In [12]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, max_len, n_segments, d_model, device):
        super(Embedding, self).__init__()
        self.tok_embed = nn.Embedding(vocab_size, d_model)  # token embedding
        self.pos_embed = nn.Embedding(max_len, d_model)      # position embedding
        self.seg_embed = nn.Embedding(n_segments, d_model)  # segment(token type) embedding
        self.norm = nn.LayerNorm(d_model)
        self.device = device

    def forward(self, x, seg):
        #x, seg: (bs, len)
        seq_len = x.size(1)
        pos = torch.arange(seq_len, dtype=torch.long).to(self.device)
        pos = pos.unsqueeze(0).expand_as(x)  # (len,) -> (bs, len)
        embedding = self.tok_embed(x) + self.pos_embed(pos) + self.seg_embed(seg)
        return self.norm(embedding)

#### 6.2 Attention mask

In [13]:
def get_attn_pad_mask(seq_q, seq_k, device):
    batch_size, len_q = seq_q.size()
    batch_size, len_k = seq_k.size()
    # eq(zero) is PAD token
    pad_attn_mask = seq_k.data.eq(0).unsqueeze(1).to(device)  # batch_size x 1 x len_k(=len_q), one is masking
    return pad_attn_mask.expand(batch_size, len_q, len_k)  # batch_size x len_q x len_k

Testing the attention mask

In [14]:
print(get_attn_pad_mask(input_ids, input_ids, device).shape)

torch.Size([16, 128, 128])


#### 6.3 Encoder

In [15]:
class EncoderLayer(nn.Module):
    def __init__(self, n_heads, d_model, d_ff, d_k, device):
        super(EncoderLayer, self).__init__()
        self.enc_self_attn = MultiHeadAttention(n_heads, d_model, d_k, device)
        self.pos_ffn       = PoswiseFeedForwardNet(d_model, d_ff)

    def forward(self, enc_inputs, enc_self_attn_mask):
        enc_outputs, attn = self.enc_self_attn(enc_inputs, enc_inputs, enc_inputs, enc_self_attn_mask) # enc_inputs to same Q,K,V
        enc_outputs = self.pos_ffn(enc_outputs) # enc_outputs: [batch_size x len_q x d_model]
        return enc_outputs, attn

In [16]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k, device):
        super(ScaledDotProductAttention, self).__init__()
        self.scale = torch.sqrt(torch.FloatTensor([d_k])).to(device)

    def forward(self, Q, K, V, attn_mask):
        scores = torch.matmul(Q, K.transpose(-1, -2)) / self.scale # scores : [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        scores.masked_fill_(attn_mask, -1e9) # Fills elements of self tensor with value where mask is one.
        attn = nn.Softmax(dim=-1)(scores)
        context = torch.matmul(attn, V)
        return context, attn 

In [17]:
n_layers = 6    # number of Encoder of Encoder Layer
n_heads  = 8    # number of heads in Multi-Head Attention
d_model  = 768  # Embedding Size
d_ff = 768 * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_model, d_k, device):
        super(MultiHeadAttention, self).__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.d_k = d_k
        self.d_v = d_k
        self.W_Q = nn.Linear(d_model, d_k * n_heads)
        self.W_K = nn.Linear(d_model, d_k * n_heads)
        self.W_V = nn.Linear(d_model, self.d_v * n_heads)
        self.device = device
    def forward(self, Q, K, V, attn_mask):
        # q: [batch_size x len_q x d_model], k: [batch_size x len_k x d_model], v: [batch_size x len_k x d_model]
        residual, batch_size = Q, Q.size(0)
        # (B, S, D) -proj-> (B, S, D) -split-> (B, S, H, W) -trans-> (B, H, S, W)
        q_s = self.W_Q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)  # q_s: [batch_size x n_heads x len_q x d_k]
        k_s = self.W_K(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1,2)  # k_s: [batch_size x n_heads x len_k x d_k]
        v_s = self.W_V(V).view(batch_size, -1, self.n_heads, self.d_v).transpose(1,2)  # v_s: [batch_size x n_heads x len_k x d_v]

        attn_mask = attn_mask.unsqueeze(1).repeat(1, self.n_heads, 1, 1) # attn_mask : [batch_size x n_heads x len_q x len_k]

        # context: [batch_size x n_heads x len_q x d_v], attn: [batch_size x n_heads x len_q(=len_k) x len_k(=len_q)]
        context, attn = ScaledDotProductAttention(self.d_k, self.device)(q_s, k_s, v_s, attn_mask)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.n_heads * self.d_v) # context: [batch_size x len_q x n_heads * d_v]
        output = nn.Linear(self.n_heads * self.d_v, self.d_model, device=self.device)(context)
        return nn.LayerNorm(self.d_model, device=self.device)(output + residual), attn # output: [batch_size x len_q x d_model]

In [19]:
class PoswiseFeedForwardNet(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PoswiseFeedForwardNet, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # (batch_size, len_seq, d_model) -> (batch_size, len_seq, d_ff) -> (batch_size, len_seq, d_model)
        return self.fc2(F.gelu(self.fc1(x)))


#### 6.4 Putting them together

In [20]:
class BERT(nn.Module):
    def __init__(self, n_layers, n_heads, d_model, d_ff, d_k, n_segments, vocab_size, max_len, device):
        super(BERT, self).__init__()
        self.params = {'n_layers': n_layers, 'n_heads': n_heads, 'd_model': d_model,
                       'd_ff': d_ff, 'd_k': d_k, 'n_segments': n_segments,
                       'vocab_size': vocab_size, 'max_len': max_len}
        self.embedding = Embedding(vocab_size, max_len, n_segments, d_model, device)
        self.layers = nn.ModuleList([EncoderLayer(n_heads, d_model, d_ff, d_k, device) for _ in range(n_layers)])
        self.fc = nn.Linear(d_model, d_model)
        self.activ = nn.Tanh()
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, 2)
        # decoder is shared with embedding layer
        embed_weight = self.embedding.tok_embed.weight
        n_vocab, n_dim = embed_weight.size()
        self.decoder = nn.Linear(n_dim, n_vocab, bias=False)
        self.decoder.weight = embed_weight
        self.decoder_bias = nn.Parameter(torch.zeros(n_vocab))
        self.device = device

    def forward(self, input_ids, segment_ids, masked_pos):
        output = self.embedding(input_ids, segment_ids)
        enc_self_attn_mask = get_attn_pad_mask(input_ids, input_ids, self.device)
        for layer in self.layers:
            output, enc_self_attn = layer(output, enc_self_attn_mask)
        # output : [batch_size, len, d_model], attn : [batch_size, n_heads, d_mode, d_model]
        
        # 1. predict next sentence
        # it will be decided by first token(CLS)
        h_pooled   = self.activ(self.fc(output[:, 0])) # [batch_size, d_model]
        logits_nsp = self.classifier(h_pooled) # [batch_size, 2]

        # 2. predict the masked token
        masked_pos = masked_pos[:, :, None].expand(-1, -1, output.size(-1)) # [batch_size, max_pred, d_model]
        h_masked = torch.gather(output, 1, masked_pos) # masking position [batch_size, max_pred, d_model]
        h_masked  = self.norm(F.gelu(self.linear(h_masked)))
        logits_lm = self.decoder(h_masked) + self.decoder_bias # [batch_size, max_pred, n_vocab]

        return logits_lm, logits_nsp
    
    def get_last_hidden_state(self, input_ids, segment_ids):
        output = self.embedding(input_ids, segment_ids)
        enc_self_attn_mask = get_attn_pad_mask(input_ids, input_ids, self.device)
        for layer in self.layers:
            output, enc_self_attn = layer(output, enc_self_attn_mask)

        return output

### Step 7: Training

In [21]:
from tqdm.auto import tqdm

n_layers = 12    # number of Encoder of Encoder Layer
n_heads  = 12    # number of heads in Multi-Head Attention
d_model  = 768  # Embedding Size
d_ff = d_model * 4  # 4*d_model, FeedForward dimension
d_k = d_v = 64  # dimension of K(=Q), V
n_segments = 2

num_epoch = 5000
model = BERT(
    n_layers, 
    n_heads, 
    d_model, 
    d_ff, 
    d_k, 
    n_segments, 
    vocab_size, 
    max_len, 
    device
).to(device)  # Move model to GPU

In [22]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [23]:
batch = make_batch()
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(*batch))

# Move inputs to GPU
input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)
masked_tokens = masked_tokens.to(device)
masked_pos = masked_pos.to(device)
isNext = isNext.to(device)

# Wrap the epoch loop with tqdm
for epoch in tqdm(range(num_epoch), desc="Training Epochs"):
    optimizer.zero_grad()
    logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)    
    #logits_lm: (bs, max_mask, vocab_size) ==> (6, 5, 34)
    #logits_nsp: (bs, yes/no) ==> (6, 2)

    #1. mlm loss
    #logits_lm.transpose: (bs, vocab_size, max_mask) vs. masked_tokens: (bs, max_mask)
    loss_lm = criterion(logits_lm.transpose(1, 2), masked_tokens) # for masked LM
    loss_lm = (loss_lm.float()).mean()
    #2. nsp loss
    #logits_nsp: (bs, 2) vs. isNext: (bs, )
    loss_nsp = criterion(logits_nsp, isNext) # for sentence classification
    
    #3. combine loss
    loss = loss_lm + loss_nsp
    if epoch % 100 == 0:
        print('Epoch:', '%02d' % (epoch), 'loss =', '{:.6f}'.format(loss))
    loss.backward()
    optimizer.step()

Training Epochs:   0%|          | 0/5000 [00:00<?, ?it/s]

Epoch: 00 loss = 140.476624
Epoch: 100 loss = 6.877973
Epoch: 200 loss = 4.018567
Epoch: 300 loss = 3.742838
Epoch: 400 loss = 3.737425
Epoch: 500 loss = 7.730536
Epoch: 600 loss = 7.466967
Epoch: 700 loss = 3.756582
Epoch: 800 loss = 3.840624
Epoch: 900 loss = 3.637219
Epoch: 1000 loss = 3.659839
Epoch: 1100 loss = 3.976768
Epoch: 1200 loss = 3.618128
Epoch: 1300 loss = 3.609130
Epoch: 1400 loss = 3.615615
Epoch: 1500 loss = 3.653230
Epoch: 1600 loss = 3.622858
Epoch: 1700 loss = 3.877427
Epoch: 1800 loss = 3.776543
Epoch: 1900 loss = 3.609907
Epoch: 2000 loss = 3.612387
Epoch: 2100 loss = 3.611027
Epoch: 2200 loss = 3.618262
Epoch: 2300 loss = 3.605427
Epoch: 2400 loss = 3.595250
Epoch: 2500 loss = 3.626322
Epoch: 2600 loss = 3.729091
Epoch: 2700 loss = 3.608747
Epoch: 2800 loss = 3.625455
Epoch: 2900 loss = 3.665984
Epoch: 3000 loss = 3.628851
Epoch: 3100 loss = 3.609589
Epoch: 3200 loss = 3.721321
Epoch: 3300 loss = 3.609958
Epoch: 3400 loss = 3.613684
Epoch: 3500 loss = 3.619444
E

In [24]:
params = {
    "n_layers": n_layers,
    "n_heads": n_heads,
    "d_model": d_model,
    "d_ff": d_ff,
    "d_k": d_k,
    "n_segments": n_segments,
    "vocab_size": vocab_size,
    "max_len": max_len
}


In [25]:
import os

os.makedirs("model", exist_ok=True)

torch.save(
    {
        "params": params,
        "state_dict": model.state_dict()
    },
    "model/bert_model.pth"
)

print("Model + params saved successfully")


Model + params saved successfully


### Step 8: Inference

In [26]:
# Predict mask tokens ans isNext
input_ids, segment_ids, masked_tokens, masked_pos, isNext = map(torch.LongTensor, zip(batch[2]))
print([id2word[w.item()] for w in input_ids[0] if id2word[w.item()] != '[PAD]'])
input_ids = input_ids.to(device)
segment_ids = segment_ids.to(device)
masked_tokens = masked_tokens.to(device)
masked_pos = masked_pos.to(device)
isNext = isNext.to(device)

logits_lm, logits_nsp = model(input_ids, segment_ids, masked_pos)
#logits_lm:  (1, max_mask, vocab_size) ==> (1, 5, 34)
#logits_nsp: (1, yes/no) ==> (1, 2)

#predict masked tokens
#max the probability along the vocab dim (2), [1] is the indices of the max, and [0] is the first value
logits_lm = logits_lm.data.cpu().max(2)[1][0].data.numpy() 
#note that zero is padding we add to the masked_tokens
print('masked tokens (words) : ',[id2word[pos.item()] for pos in masked_tokens[0]])
print('masked tokens list : ',[pos.item() for pos in masked_tokens[0]])
print('masked tokens (words) : ',[id2word[pos.item()] for pos in logits_lm])
print('predict masked tokens list : ', [pos for pos in logits_lm])

#predict nsp
logits_nsp = logits_nsp.cpu().data.max(1)[1][0].data.numpy()
print(logits_nsp)
print('isNext : ', True if isNext else False)
print('predict isNext : ',True if logits_nsp else False)

['[CLS]', '[MASK]', 'song', 'was', '[MASK]', 'by', 'pitchfork', 'media', 'as', 'their', 'number', 'one', 'song', 'of', '2006', 'also', 'ranking', 'the', 'song', 'at', '26', 'on', '"', 'the', 'top', '[MASK]', 'tracks', 'of', 'the', '2000s', '"', '"', 'at', 'the', '49th', 'grammy', 'awards', '"', 'my', 'love', '"', 'won', 'a', '[MASK]', '[MASK]', 'in', 'the', 'category', '[MASK]', '[MASK]', 'rap', '[SEP]', '[MASK]', 'sam', 'malone', '(', 'ted', 'danson', ')', 'receives', 'telephone', 'calls', 'from', 'women', 'whom', 'he', 'previously', 'dated', '[MASK]', 'they', 'are', 'angry', 'with', 'him', 'for', 'making', 'dates', 'and', 'not', '[MASK]', 'he', 'eventually', 'discovers', 'that', 'his', '[MASK]', 'little', '[MASK]', 'book', '[MASK]', 'has', 'been', '[MASK]', 'and', 'enrolls', 'the', 'help', 'of', 'bar', '[MASK]', 'to', 'find', '[SEP]']
masked tokens (words) :  ['black', 'best', 'award', '"', 'picked', 'the', 'stolen', ';', 'grammy', '500', 'patrons', 'arriving', '"', 'meanwhile', 'of'